# 🎮 Roll Compactor Control Performance: PID Analysis & SPC

**Dataset:** Roll Compactor Control Performance (Synthetic) v1.0  
**Publisher:** [Innovative Process Applications (IPA)](https://www.innovativeprocess.com)  
**License:** CC BY 4.0  
**Scientific basis:** Szappanos-Csordás (2018), Chapter 3.1

> ⚠️ **Synthetic educational data** — not real measurements.

---

This notebook covers:
1. Load both summary and time-series data
2. Visualize PID step responses across control architectures
3. Compare control quality metrics (CV%, settling time, overshoot)
4. Build SPC control charts
5. Quantify the twin feed screw advantage
6. Classify control quality from time-series features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load data
df_summary = pd.read_csv('control_performance_summary_v1.0.csv')
df_ts = pd.read_csv('control_performance_timeseries_v1.0.csv')

print(f'Summary: {df_summary.shape[0]} runs, {df_summary.shape[1]} columns')
print(f'Time series: {df_ts.shape[0]} rows, {df_ts.shape[1]} columns')
print(f'Control architectures: {df_summary["control_architecture"].unique()}')
df_summary.head()

## Part 1: PID Step Response Visualization

The defining characteristic of a PID controller is its step response — how
the actual value tracks a setpoint change. We’ll compare the four control
architectures responding to the same SCF step-up scenario.

In [ ]:
# Pick one material and one scenario to compare architectures
scenario = 'scf_step_up'
material = 'MCC_Mannitol_Mix'

architectures = ['no_gap_control', 'pid_gw_screw', 'pid_scf_gw', 'pid_scf_gw_twin_screw']
labels = ['No Gap Control', 'PID: GW+Screw', 'PID: SCF+GW', 'PID: SCF+GW+Twin Screw']
colors = ['#cc4444', '#d4a017', '#2E86C1', '#008080']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for arch, label, color in zip(architectures, labels, colors):
    run = df_summary[(df_summary['control_architecture']==arch) &
                     (df_summary['material']==material) &
                     (df_summary['scenario']==scenario)]
    if len(run) == 0:
        continue
    rid = run.iloc[0]['run_id']
    ts = df_ts[df_ts['run_id']==rid]

    axes[0].plot(ts['time_s'], ts['scf_actual_kN_per_cm'], label=label,
                color=color, alpha=0.8, linewidth=1.2)
    axes[1].plot(ts['time_s'], ts['gw_actual_mm'], label=label,
                color=color, alpha=0.8, linewidth=1.2)

# Setpoint lines
axes[0].axhline(4.0, color='gray', linestyle=':', alpha=0.5)
axes[0].axhline(8.0, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(30, color='black', linestyle='--', alpha=0.3, label='Setpoint change')
axes[0].set_ylabel('SCF (kN/cm)')
axes[0].set_title(f'PID Step Response Comparison \u2014 SCF Step Up (4\u21928 kN/cm), {material}')
axes[0].legend(loc='lower right', fontsize=9)

axes[1].axvline(30, color='black', linestyle='--', alpha=0.3)
axes[1].set_ylabel('Gap Width (mm)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Gap Width Response During SCF Step Change')
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print('Key observations:')
print('- No gap control: slow approach, steady-state offset, noisy')
print('- PID SCF+GW: overshoot then clean settling')
print('- Twin screw: tightest band, fastest settling, least noise')

## Part 2: Control Quality Metrics by Architecture

CV% (coefficient of variation) is the primary metric from Chapter 3.1 of
the dissertation. Lower CV = more robust process.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

order = ['no_gap_control', 'pid_gw_screw', 'pid_scf_gw', 'pid_scf_gw_twin_screw']
palette = ['#cc4444', '#d4a017', '#2E86C1', '#008080']

sns.boxplot(data=df_summary, x='control_architecture', y='scf_ss_cv_pct',
            order=order, palette=palette, ax=axes[0])
axes[0].set_title('SCF Steady-State CV%')
axes[0].set_ylabel('CV %')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_xlabel('')

sns.boxplot(data=df_summary, x='control_architecture', y='gw_ss_cv_pct',
            order=order, palette=palette, ax=axes[1])
axes[1].set_title('Gap Width Steady-State CV%')
axes[1].set_ylabel('CV %')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_xlabel('')

sns.boxplot(data=df_summary, x='control_architecture', y='scf_settling_time_s',
            order=order, palette=palette, ax=axes[2])
axes[2].set_title('SCF Settling Time')
axes[2].set_ylabel('Time (s)')
axes[2].tick_params(axis='x', rotation=30)
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

print('\nMean CV% by architecture:')
print(df_summary.groupby('control_architecture')[['scf_ss_cv_pct','gw_ss_cv_pct']].mean().round(2).loc[order])

## Part 3: Material Effect on Controllability

Brittle materials (mannitol) produce more erratic force signals due to
particle fragmentation, making the control task harder.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mat_order = ['MCC_101', 'MCC_Mannitol_Mix', 'Mannitol_SD']
mat_colors = ['#008080', '#2E86C1', '#d4a017']

sns.boxplot(data=df_summary, x='material', y='scf_ss_cv_pct',
            order=mat_order, palette=mat_colors, ax=axes[0])
axes[0].set_title('SCF CV% by Material')
axes[0].set_ylabel('SCF CV %')
axes[0].set_xlabel('')

sns.boxplot(data=df_summary, x='material', y='gw_ss_cv_pct',
            order=mat_order, palette=mat_colors, ax=axes[1])
axes[1].set_title('Gap Width CV% by Material')
axes[1].set_ylabel('GW CV %')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print('Interpretation:')
print('- MCC (plastic): smoothest compaction, easiest to control')
print('- Mannitol (brittle): particle fragmentation creates force spikes')
print('- Mixture: intermediate behavior')

## Part 4: SPC Control Charts

Statistical Process Control charts are standard tools in pharma
manufacturing. Let’s build X-bar and R charts for a sample run.

In [ ]:
# Pick the twin-screw + MCC steady-state baseline for the cleanest signal
run_spc = df_summary[(df_summary['control_architecture']=='pid_scf_gw_twin_screw') &
                     (df_summary['material']=='MCC_101') &
                     (df_summary['scenario']=='steady_state_baseline')]
if len(run_spc) > 0:
    rid = run_spc.iloc[0]['run_id']
    ts = df_ts[df_ts['run_id']==rid].copy()

    # Use only steady-state portion (t > 30s)
    ts_ss = ts[ts['time_s'] >= 30.0].copy()

    scf_mean = ts_ss['scf_actual_kN_per_cm'].mean()
    scf_std = ts_ss['scf_actual_kN_per_cm'].std()
    ucl = scf_mean + 3 * scf_std
    lcl = scf_mean - 3 * scf_std

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(ts_ss['time_s'], ts_ss['scf_actual_kN_per_cm'],
            'o-', markersize=3, color='#008080', alpha=0.7)
    ax.axhline(scf_mean, color='#1B2A3B', linewidth=2, label=f'CL = {scf_mean:.3f}')
    ax.axhline(ucl, color='red', linestyle='--', label=f'UCL = {ucl:.3f}')
    ax.axhline(lcl, color='red', linestyle='--', label=f'LCL = {lcl:.3f}')
    ax.fill_between(ts_ss['time_s'], lcl, ucl, alpha=0.05, color='red')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('SCF (kN/cm)')
    ax.set_title(f'X-bar Control Chart \u2014 Twin Screw PID, MCC 101, Steady State')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    print(f'Process capability summary:')
    print(f'  Mean: {scf_mean:.3f} kN/cm')
    print(f'  Std:  {scf_std:.4f} kN/cm')
    print(f'  CV:   {scf_std/scf_mean*100:.2f}%')
    spec_half = 0.5  # example spec: setpoint +/- 0.5 kN/cm
    cpk = min(ucl - scf_mean, scf_mean - lcl) / (3 * scf_std)
    print(f'  Cpk (\u00b13\u03c3 natural): {cpk:.2f}')

## Part 5: Twin Feed Screw Advantage

This is the key IPA design differentiator. Twin feed screws provide more
uniform powder delivery, which directly reduces process variability.

In [ ]:
# Compare single vs twin screw (both with SCF+GW PID)
single = df_summary[df_summary['control_architecture']=='pid_scf_gw']
twin = df_summary[df_summary['control_architecture']=='pid_scf_gw_twin_screw']

metrics = ['scf_ss_cv_pct', 'gw_ss_cv_pct', 'scf_settling_time_s', 'scf_overshoot_pct']
metric_labels = ['SCF CV%', 'GW CV%', 'Settling Time (s)', 'Overshoot %']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric, mlabel in zip(axes, metrics, metric_labels):
    data = pd.DataFrame({
        'Single Screw': single[metric].values,
        'Twin Screw': twin[metric].values,
    })
    data.plot.box(ax=ax, color=dict(boxes='#008080', whiskers='gray',
                                     medians='#1B2A3B', caps='gray'))
    ax.set_title(mlabel)
    ax.set_ylabel(mlabel)

plt.suptitle('Single Screw vs. Twin Feed Screw (Same PID Architecture)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nImprovement with twin screw:')
for metric, mlabel in zip(metrics, metric_labels):
    s = single[metric].mean()
    t = twin[metric].mean()
    improvement = (s - t) / s * 100 if s > 0 else 0
    print(f'  {mlabel}: {s:.2f} \u2192 {t:.2f} ({improvement:+.1f}% improvement)')

## Part 6: Classify Control Quality from Metrics

Can we predict the control quality grade from the summary metrics? This
demonstrates how machine learning can support process monitoring.

In [ ]:
# Encode targets and features
le = LabelEncoder()
y = le.fit_transform(df_summary['control_quality_grade'])

feature_cols = ['scf_ss_cv_pct', 'gw_ss_cv_pct', 'scf_deviation_from_setpoint_pct',
                'gw_deviation_from_setpoint_pct', 'scf_settling_time_s',
                'gw_settling_time_s', 'scf_overshoot_pct']
X = df_summary[feature_cols].values

rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'RF 5-fold CV accuracy: {scores.mean():.3f} \u00b1 {scores.std():.3f}')

rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot.barh(color='#008080', ax=ax)
ax.set_title('Feature Importance for Control Quality Grade Prediction')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Takeaways

1. **Control architecture matters enormously:** Going from no gap control to
   full PID SCF+GW control reduces process variability by 50–70%. This is
   not optional in regulated pharma manufacturing.

2. **Twin feed screws are a multiplicative advantage:** On top of PID control,
   twin screws provide an additional 20–30% reduction in CV by smoothing
   feed rate fluctuations. This is a mechanical design choice, not a tuning
   parameter — it must be built into the machine.

3. **Material properties affect control difficulty:** Brittle materials
   produce noisier signals that challenge even the best PID controllers.
   Process development must account for this.

4. **SPC tools apply directly to continuous granulation:** Control charts,
   Cp/Cpk, and trend analysis are standard pharma quality tools that
   connect naturally to roll compaction process data.

---

For twin-feed-screw roller compactors with advanced PID control and direct
engineer support, see IPA’s CL-series compactors:
**https://www.innovativeprocess.com**

*Dataset © 2026 Innovative Process Applications, CC BY 4.0.*  
*Scientific basis: Szappanos-Csordás (2018), Chapter 3.1.*